# 02 — Linear Regression: Learning a Line from Data

## The Big Idea

We have a cloud of points `(x, y)`.  
We want to find the **straight line** `y = w·x + b` that fits them best.

Two numbers control the line:
- **w** (weight / slope) — how steeply the line rises
- **b** (bias / intercept) — where the line crosses the y-axis

We start with a random guess, measure how wrong we are, then nudge  
the line in the right direction — over and over.  That process is called  
**gradient descent**, and it is the engine behind all of modern AI.

## Step 1 — Generate & Visualise the Data

We create 30 points that follow `y = 10·x + 30` with a bit of random noise  
added — like real-world measurements.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

np.random.seed(42)
x = np.linspace(0, 10, 30)          # 30 evenly-spaced x values from 0 to 10
y = 10 * x + 30 + np.random.randn(30) * 8   # true line + noise

plt.figure(figsize=(8, 4))
plt.scatter(x, y, color="steelblue", zorder=3, label="data points")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Our Dataset")
plt.legend()
plt.tight_layout()
plt.show()

## Step 2 — Start with a Flat Line (w = 0, b = 0)

Before learning anything our model predicts `y = 0` for every `x`.  
Plotting it on top of the data shows just how wrong we start.

In [ ]:
w = 0.0   # slope — starts flat
b = 0.0   # intercept — starts at zero

x_line = np.array([0, 10])
y_line = w * x_line + b

plt.figure(figsize=(8, 4))
plt.scatter(x, y, color="steelblue", zorder=3, label="data points")
plt.plot(x_line, y_line, color="red", linewidth=2, label=f"initial line  w={w}, b={b}")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Data + Initial (terrible) Line")
plt.legend()
plt.tight_layout()
plt.show()

print("The red line is completely flat — it hasn't learned anything yet.")

## Step 3 — Measuring How Wrong We Are (the Loss Function)

For each data point we compute the **error**: predicted value minus true value.

We square each error (to make negatives positive and punish big errors more),  
then take the average.  This is called **Mean Squared Error (MSE)**:

```
MSE = average( (predicted - actual)² )
```

A perfect fit has MSE = 0.  A terrible fit has a huge MSE.

In [ ]:
def compute_loss(w, b, x, y):
    predictions = w * x + b
    errors = predictions - y
    return (errors ** 2).mean()

initial_loss = compute_loss(0.0, 0.0, x, y)
print(f"Loss with flat line (w=0, b=0): {initial_loss:.1f}")
avg_error = initial_loss ** 0.5
print(f"Each data point is off by roughly: {avg_error:.1f} units on average")

## Step 4 — Gradient Descent: Nudging the Line in the Right Direction

The key question: **which way should we move `w` and `b`?**

We compute the **gradient** — how much the loss changes if we  
increase `w` or `b` slightly.  Then we move in the *opposite* direction  
(downhill on the loss surface).

### The gradient formulas (from calculus)

After differentiating MSE with respect to `w` and `b`:

```
grad_w = (2/n) * sum( errors * x )   ← each error is weighted by x
grad_b = (2/n) * sum( errors )        ← b shifts all predictions equally
```

Then we take a small step:

```
w ← w - learning_rate * grad_w
b ← b - learning_rate * grad_b
```

The **learning rate** controls step size. Too large → overshoots;  
too small → learns very slowly.  We'll use `0.01`.

## Step 5 — Training Loop: Watch the Line Move

We run 200 update steps and capture the line at key moments  
so we can watch it converge toward the true relationship.

In [ ]:
w = 0.0
b = 0.0
learning_rate = 0.01
n = len(x)
loss_history = []

# Steps at which we'll take a snapshot of the line (1-indexed step numbers)
snapshot_steps = {1, 5, 20, 50, 100, 200}
snapshots = []   # list of (step_number, w, b)

for step in range(200):
    predictions = w * x + b
    errors = predictions - y          # positive → predicted too high

    # Gradient of w: weight each error by how much x contributed
    grad_w = (2 / n) * (errors * x).sum()

    # Gradient of b: b shifts every prediction by the same amount
    grad_b = (2 / n) * errors.sum()

    # Move opposite to the gradient
    w -= learning_rate * grad_w
    b -= learning_rate * grad_b

    loss_history.append(compute_loss(w, b, x, y))

    step_number = step + 1   # convert 0-indexed to human-readable
    if step_number in snapshot_steps:
        snapshots.append((step_number, w, b))

# ---- Plot the snapshots ----
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(x, y, color="steelblue", zorder=3, label="data")

colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(snapshots)))
x_line = np.array([0, 10])
for (step, ws, bs), color in zip(snapshots, colors):
    y_line = ws * x_line + bs
    ax.plot(x_line, y_line, color=color, linewidth=2, label=f"step {step}  (w={ws:.1f}, b={bs:.1f})")

ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Line evolving over 200 training steps")
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

## Step 6 — Final Result & Loss Curve

How close did we get to the true parameters `w=10, b=30`?

In [ ]:
print(f'Learned:  y = {w:.2f} * x + {b:.2f}')
print(f'Truth:    y = 10.00 * x + 30.00')
print(f'Final loss: {loss_history[-1]:.2f}')

plt.figure(figsize=(8, 3))
plt.plot(loss_history, color="crimson")
plt.xlabel("Training step")
plt.ylabel("MSE loss")
plt.title("Loss curve — shrinking as the model learns")
plt.tight_layout()
plt.show()

## Key Lessons

1. **w and b** are the only two numbers our model learns — yet that's enough  
   to capture a linear relationship.
2. **Gradient descent** uses calculus to figure out which direction to nudge  
   the parameters — no guessing needed.
3. The **loss curve** always decreasing is a healthy sign; if it bounces around  
   the learning rate is probably too high.
4. This same mechanism — compute loss → compute gradients → update parameters —  
   is used in **every** neural network, including GPT.